# Structured Extraction from PDFs with Validation Loops

PDFs are the format enterprises use for invoices, contracts, medical records, and financial reports. They are also the format most likely to break naive extraction pipelines: rotated text, merged cells in tables, inconsistent decimal separators, currency symbols that confuse parsers, and pages that scan poorly.

A single-pass extraction — send image to model, parse JSON response, write to database — fails silently in production. The model returns a plausible-looking result with `null` fields or hallucinated totals. The pipeline completes with no error. The database is quietly wrong.

This notebook teaches a **validate-and-retry loop**: extract, check, and if the check fails, feed the specific error back to the model with an instruction to fix it. Three attempts maximum. If confidence is still low after three attempts, flag for human review.

## What you will build

```
PDF page (base64 image)
        │
        ▼
┌─────────────────┐
│  GPT-4o Vision  │  Attempt 1: extract → InvoiceData (Pydantic)
└────────┬────────┘
         │ structured JSON
         ▼
┌─────────────────────────────────┐
│  Validation Layer (3 checks)    │
│  1. Schema: required fields     │
│  2. Type: amount parseable?     │
│  3. Business: line items sum?   │
└────────┬────────────────────────┘
         │
    Pass? ──YES──► return result
         │
        FAIL
         │
         ▼
┌─────────────────┐
│  Correction     │  Feed error message back to GPT-4o
│  Prompt         │  "Please fix: {errors}"  (max 3 attempts)
└────────┬────────┘
         │
         └──► repeat validation
         │
    confidence < 3 after 3 tries → HUMAN REVIEW queue
```

For multi-page documents, all pages run through `asyncio.gather` in parallel.

## Prerequisites

- An OpenAI API key set as `OPENAI_API_KEY` in your environment or a `.env` file.
- Python 3.10+, `openai>=1.30`, `pydantic>=2.0`, `matplotlib` (for synthetic invoice generation).

## Setup

Install dependencies and configure the OpenAI client.

In [ ]:
%pip install openai pydantic python-dotenv matplotlib --quiet

In [ ]:
import asyncio
import base64
import io
import json
import os
import re
import time
from dataclasses import dataclass, field
from datetime import datetime
from textwrap import dedent
from typing import Optional

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from openai import AsyncOpenAI
from pydantic import BaseModel, ValidationError, field_validator, model_validator

load_dotenv()

client = AsyncOpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

EXTRACTION_MODEL = "gpt-4o"       # Vision + structured output extraction
VALIDATION_MODEL = "gpt-4o-mini"  # Lightweight second-opinion validation
MAX_ATTEMPTS = 3
CONFIDENCE_THRESHOLD = 3  # Flag for human review if score < this

print(f"Extraction model  : {EXTRACTION_MODEL}")
print(f"Validation model  : {VALIDATION_MODEL}")
print(f"Max retry attempts: {MAX_ATTEMPTS}")

## Step 1 — Define the Extraction Schema

Use Pydantic v2 to define the target schema. The model will be prompted to return JSON matching this shape. Pydantic gives us free schema documentation, type coercion, and a clean place to add field-level validators.

We use `InvoiceLineItem` for individual rows and `InvoiceData` for the document. Note the confidence field: GPT-4o will self-report how certain it is about the extraction on a 1–5 scale.

In [ ]:
class InvoiceLineItem(BaseModel):
    description: str
    quantity: float
    unit_price: float
    line_total: float


class InvoiceData(BaseModel):
    vendor: str
    invoice_number: Optional[str] = None
    date: str                        # ISO 8601 preferred: YYYY-MM-DD
    due_date: Optional[str] = None
    currency: str = "USD"
    subtotal: Optional[float] = None
    tax: Optional[float] = None
    total_amount: float
    line_items: list[InvoiceLineItem]
    confidence: int                  # 1 (very uncertain) to 5 (very confident)
    extraction_notes: Optional[str] = None  # model explains any ambiguity

    @field_validator("date", "due_date", mode="before")
    @classmethod
    def parse_date_field(cls, v: Optional[str]) -> Optional[str]:
        """Accept several common date formats, normalize to YYYY-MM-DD."""
        if v is None:
            return None
        for fmt in ("%Y-%m-%d", "%m/%d/%Y", "%d/%m/%Y", "%B %d, %Y", "%b %d, %Y"):
            try:
                return datetime.strptime(v.strip(), fmt).strftime("%Y-%m-%d")
            except ValueError:
                continue
        raise ValueError(f"Cannot parse date: {v!r}. Expected formats: YYYY-MM-DD, MM/DD/YYYY, Month DD YYYY")

    @field_validator("confidence")
    @classmethod
    def confidence_range(cls, v: int) -> int:
        if not 1 <= v <= 5:
            raise ValueError(f"confidence must be 1–5, got {v}")
        return v


# Show the schema GPT-4o will be asked to produce
print(json.dumps(InvoiceData.model_json_schema(), indent=2))

## Step 2 — Create a Synthetic Invoice Image

Rather than requiring you to supply a real PDF, we generate a realistic invoice using `matplotlib` and encode it as base64. This produces a PNG image that resembles a scanned document — the same format the vision API receives when you convert PDF pages to images.

The synthetic invoice contains intentional realism: a tax line, per-item prices, and a total that must equal the sum of line items. We will use these known values to verify that the extraction is arithmetically correct.

In [ ]:
# Ground-truth values we embed in the invoice image.
# The validation layer will check that extracted totals match.
INVOICE_GROUND_TRUTH = {
    "vendor": "Acme Software Solutions Inc.",
    "invoice_number": "INV-2024-0087",
    "date": "2024-03-15",
    "due_date": "2024-04-14",
    "line_items": [
        {"description": "Professional Services - API Integration", "qty": 40, "unit": 150.00, "total": 6000.00},
        {"description": "Cloud Infrastructure Setup",             "qty": 1,  "unit": 850.00, "total": 850.00},
        {"description": "SaaS License (Annual)",                 "qty": 5,  "unit": 299.00, "total": 1495.00},
    ],
    "subtotal": 8345.00,
    "tax": 667.60,       # 8% of subtotal
    "total": 9012.60,
}


def generate_invoice_image() -> str:
    """Render a synthetic invoice as a PNG and return it as a base64 string."""
    fig, ax = plt.subplots(figsize=(8.5, 11))
    ax.set_xlim(0, 8.5)
    ax.set_ylim(0, 11)
    ax.axis("off")
    fig.patch.set_facecolor("white")

    gt = INVOICE_GROUND_TRUTH

    # ── Header ──────────────────────────────────────────────────────────────
    ax.text(4.25, 10.5, gt["vendor"], ha="center", fontsize=14, fontweight="bold", color="#1a237e")
    ax.text(4.25, 10.2, "123 Tech Park, San Francisco, CA 94107", ha="center", fontsize=9, color="#555")
    ax.text(4.25, 10.0, "Tel: (415) 555-0190  |  billing@acmesoftware.example.com", ha="center", fontsize=9, color="#555")

    ax.add_patch(mpatches.FancyBboxPatch((0.3, 9.5), 7.9, 0.35, boxstyle="round,pad=0.05",
                                          facecolor="#1a237e", edgecolor="none"))
    ax.text(4.25, 9.63, "INVOICE", ha="center", fontsize=16, fontweight="bold", color="white")

    # ── Invoice metadata ─────────────────────────────────────────────────────
    ax.text(0.5, 9.3, f"Invoice #: {gt['invoice_number']}",  fontsize=10)
    ax.text(0.5, 9.1, f"Date:      {gt['date']}",            fontsize=10)
    ax.text(0.5, 8.9, f"Due Date:  {gt['due_date']}",        fontsize=10)
    ax.text(4.5, 9.3, "Bill To:",                            fontsize=10, fontweight="bold")
    ax.text(4.5, 9.1, "Globex Corporation",                  fontsize=10)
    ax.text(4.5, 8.9, "742 Evergreen Terrace, Springfield",  fontsize=10)

    # ── Table header ─────────────────────────────────────────────────────────
    ax.add_patch(mpatches.Rectangle((0.3, 8.5), 7.9, 0.3, facecolor="#e8eaf6", edgecolor="#9fa8da"))
    ax.text(0.5,  8.62, "Description",  fontsize=9, fontweight="bold")
    ax.text(5.0,  8.62, "Qty",          fontsize=9, fontweight="bold", ha="center")
    ax.text(6.0,  8.62, "Unit Price",   fontsize=9, fontweight="bold", ha="center")
    ax.text(7.8,  8.62, "Total",        fontsize=9, fontweight="bold", ha="right")

    # ── Line items ───────────────────────────────────────────────────────────
    row_colors = ["#ffffff", "#f5f5f5"]
    y = 8.5
    for i, item in enumerate(gt["line_items"]):
        y -= 0.35
        ax.add_patch(mpatches.Rectangle((0.3, y), 7.9, 0.35,
                                         facecolor=row_colors[i % 2], edgecolor="#e0e0e0"))
        ax.text(0.5,  y + 0.12, item["description"],    fontsize=8.5)
        ax.text(5.0,  y + 0.12, str(item["qty"]),        fontsize=8.5, ha="center")
        ax.text(6.0,  y + 0.12, f"${item['unit']:,.2f}",  fontsize=8.5, ha="center")
        ax.text(7.8,  y + 0.12, f"${item['total']:,.2f}", fontsize=8.5, ha="right")

    # ── Totals ────────────────────────────────────────────────────────────────
    y -= 0.5
    ax.plot([5.0, 8.1], [y + 0.35, y + 0.35], color="#9fa8da", linewidth=0.8)
    ax.text(5.5, y + 0.15, "Subtotal:",  fontsize=9)
    ax.text(7.8, y + 0.15, f"${gt['subtotal']:,.2f}", fontsize=9, ha="right")
    y -= 0.3
    ax.text(5.5, y + 0.15, "Tax (8%):",  fontsize=9)
    ax.text(7.8, y + 0.15, f"${gt['tax']:,.2f}",      fontsize=9, ha="right")
    y -= 0.35
    ax.add_patch(mpatches.Rectangle((5.0, y), 3.1, 0.32, facecolor="#1a237e", edgecolor="none"))
    ax.text(5.5, y + 0.09, "TOTAL DUE:", fontsize=10, fontweight="bold", color="white")
    ax.text(7.8, y + 0.09, f"${gt['total']:,.2f}", fontsize=10, fontweight="bold", color="white", ha="right")

    # ── Footer ────────────────────────────────────────────────────────────────
    ax.text(4.25, 0.8, "Payment Terms: Net 30  |  Wire transfer or ACH accepted",
            ha="center", fontsize=8, color="#777")
    ax.text(4.25, 0.5, "Thank you for your business!",
            ha="center", fontsize=9, style="italic", color="#1a237e")

    buf = io.BytesIO()
    plt.savefig(buf, format="png", dpi=150, bbox_inches="tight", facecolor="white")
    plt.close()
    buf.seek(0)
    return base64.b64encode(buf.read()).decode("utf-8")


invoice_b64 = generate_invoice_image()
print(f"Invoice image generated: {len(invoice_b64):,} base64 chars ({len(invoice_b64)*3//4/1024:.1f} KB)")

# Preview the image inline
from IPython.display import Image as IPImage, display
display(IPImage(data=base64.b64decode(invoice_b64), width=600))

## Step 3 — The Extraction Prompt

The extraction call sends the base64 image to GPT-4o using the vision API. We embed the full Pydantic JSON schema in the system prompt so the model knows exactly what fields to produce. The model is also instructed to self-report a `confidence` score, which becomes our signal for whether to flag the result for human review.

On correction attempts, we append the specific validation errors — not the entire schema again — so the model knows exactly what it got wrong.

In [ ]:
EXTRACTION_SYSTEM = dedent("""
    You are a document extraction specialist. Extract structured invoice data
    from the provided image and return it as valid JSON matching the schema below.

    SCHEMA:
    {schema}

    RULES:
    - Return ONLY valid JSON. No markdown fences, no commentary.
    - Dates must be YYYY-MM-DD format.
    - All monetary amounts must be plain floats (e.g. 1495.00, not "$1,495.00").
    - confidence: integer 1-5. Use 5 only when every field is clearly legible.
      Use 3 when some fields are inferred. Use 1-2 when the image is too poor
      to extract reliably.
    - extraction_notes: explain any ambiguous fields or assumptions made.
    - Include ALL line items visible in the invoice.
""").strip()

CORRECTION_PREFIX = dedent("""
    Your previous extraction had validation errors. Please fix ONLY the fields
    listed below and return the corrected full JSON.

    ERRORS TO FIX:
    {errors}

    Return the corrected JSON. All other fields should remain the same.
""").strip()


async def extract_invoice(
    image_b64: str,
    previous_json: Optional[str] = None,
    errors: Optional[list[str]] = None,
) -> tuple[dict, float]:
    """
    Call GPT-4o to extract (or correct) invoice data from a base64 image.

    Args:
        image_b64:     Base64-encoded PNG of the invoice page.
        previous_json: JSON string from the prior failed attempt (for correction).
        errors:        List of validation error messages from the prior attempt.

    Returns:
        (parsed_dict, elapsed_seconds)
    """
    schema_str = json.dumps(InvoiceData.model_json_schema(), indent=2)
    system = EXTRACTION_SYSTEM.format(schema=schema_str)

    # Build the user message content
    content: list[dict] = []

    if errors and previous_json:
        error_block = "\n".join(f"- {e}" for e in errors)
        correction_text = CORRECTION_PREFIX.format(errors=error_block)
        correction_text += f"\n\nPREVIOUS JSON:\n{previous_json}"
        content.append({"type": "text", "text": correction_text})
    else:
        content.append({"type": "text", "text": "Extract all invoice data from this image."})

    content.append({
        "type": "image_url",
        "image_url": {"url": f"data:image/png;base64,{image_b64}", "detail": "high"},
    })

    t0 = time.perf_counter()
    response = await client.chat.completions.create(
        model=EXTRACTION_MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user",   "content": content},
        ],
        temperature=0.0,   # deterministic extraction
        max_tokens=2048,
        response_format={"type": "json_object"},
    )
    elapsed = time.perf_counter() - t0

    raw = response.choices[0].message.content
    return json.loads(raw), elapsed

## Step 4 — The Validation Layer

Three validators run in sequence. Each returns a list of error strings. An empty list means the check passed.

**Validator 1 — Schema check**: Did Pydantic accept the JSON? This catches missing required fields and type mismatches (e.g. `total_amount` is a string instead of a float).

**Validator 2 — Type/date check**: Can the date string be parsed? Is `total_amount` a positive number? Pydantic handles most of this via `field_validator`, but we add explicit business-logic assertions here.

**Validator 3 — Business rule check**: Do the `line_items` sum to `total_amount`? A model that reads tables poorly might get individual cells right but compute the wrong total. This catches arithmetic errors that schema validation cannot.

In [ ]:
@dataclass
class ValidationResult:
    passed: bool
    errors: list[str] = field(default_factory=list)
    invoice: Optional[InvoiceData] = None


def validate_schema(raw: dict) -> ValidationResult:
    """Validator 1: Pydantic schema validation."""
    try:
        invoice = InvoiceData(**raw)
        return ValidationResult(passed=True, invoice=invoice)
    except ValidationError as exc:
        errors = []
        for e in exc.errors():
            loc = " → ".join(str(x) for x in e["loc"])
            errors.append(f"[schema] {loc}: {e['msg']}")
        return ValidationResult(passed=False, errors=errors)


def validate_types(invoice: InvoiceData) -> ValidationResult:
    """Validator 2: Type and sanity checks beyond Pydantic defaults."""
    errors = []

    # Total must be positive
    if invoice.total_amount <= 0:
        errors.append(f"[type] total_amount must be > 0, got {invoice.total_amount}")

    # Currency must be a 3-letter code
    if not re.match(r"^[A-Z]{3}$", invoice.currency):
        errors.append(f"[type] currency must be a 3-letter ISO code, got {invoice.currency!r}")

    # Confidence must be in range (Pydantic catches this too, belt-and-suspenders)
    if not 1 <= invoice.confidence <= 5:
        errors.append(f"[type] confidence must be 1–5, got {invoice.confidence}")

    # Line item quantities and prices must be positive
    for i, item in enumerate(invoice.line_items):
        if item.quantity <= 0:
            errors.append(f"[type] line_items[{i}].quantity must be > 0, got {item.quantity}")
        if item.unit_price < 0:
            errors.append(f"[type] line_items[{i}].unit_price must be >= 0, got {item.unit_price}")

    return ValidationResult(passed=len(errors) == 0, errors=errors, invoice=invoice)


def validate_business_rules(invoice: InvoiceData, tolerance: float = 0.02) -> ValidationResult:
    """
    Validator 3: Business logic checks.

    Checks that line item row totals are internally consistent
    (qty * unit_price ≈ line_total) and that the invoice total is
    reachable from the line items (± tolerance for tax rounding).
    """
    errors = []

    # Check each line item: qty * unit_price should equal line_total
    for i, item in enumerate(invoice.line_items):
        expected = round(item.quantity * item.unit_price, 2)
        if abs(expected - item.line_total) > tolerance:
            errors.append(
                f"[business] line_items[{i}] ({item.description!r}): "
                f"qty {item.quantity} × unit_price {item.unit_price} = {expected:.2f} "
                f"but line_total is {item.line_total:.2f} (diff={abs(expected - item.line_total):.2f})"
            )

    # Check that sum of line totals ≈ subtotal (if provided) or total
    line_sum = round(sum(item.line_total for item in invoice.line_items), 2)

    if invoice.subtotal is not None:
        if abs(line_sum - invoice.subtotal) > tolerance:
            errors.append(
                f"[business] line items sum to {line_sum:.2f} "
                f"but subtotal is {invoice.subtotal:.2f} "
                f"(diff={abs(line_sum - invoice.subtotal):.2f})"
            )
        # subtotal + tax should ≈ total; if tax is omitted, subtotal should ≈ total
        if invoice.tax is not None:
            computed_total = round(invoice.subtotal + invoice.tax, 2)
            if abs(computed_total - invoice.total_amount) > tolerance:
                errors.append(
                    f"[business] subtotal {invoice.subtotal:.2f} + tax {invoice.tax:.2f} "
                    f"= {computed_total:.2f} but total_amount is {invoice.total_amount:.2f}"
                )
        else:
            # Tax absent: subtotal should equal total_amount directly.
            if abs(invoice.subtotal - invoice.total_amount) > tolerance:
                errors.append(
                    f"[business] tax omitted but subtotal {invoice.subtotal:.2f} "
                    f"!= total_amount {invoice.total_amount:.2f} "
                    f"(diff={abs(invoice.subtotal - invoice.total_amount):.2f})"
                )
    else:
        # No subtotal: when tax is present compare line_sum + tax to total_amount;
        # otherwise fall back to a ±20% sanity check.
        if invoice.tax is not None:
            computed_total = round(line_sum + invoice.tax, 2)
            if abs(computed_total - invoice.total_amount) > tolerance:
                errors.append(
                    f"[business] line items ({line_sum:.2f}) + tax ({invoice.tax:.2f}) "
                    f"= {computed_total:.2f} but total_amount is {invoice.total_amount:.2f} "
                    f"(diff={abs(computed_total - invoice.total_amount):.2f})"
                )
        elif abs(line_sum - invoice.total_amount) > invoice.total_amount * 0.20:
            errors.append(
                f"[business] line items sum to {line_sum:.2f} "
                f"but total_amount is {invoice.total_amount:.2f} — "
                f"large discrepancy ({abs(line_sum - invoice.total_amount) / invoice.total_amount * 100:.1f}%)"
            )

    return ValidationResult(passed=len(errors) == 0, errors=errors, invoice=invoice)


def run_all_validators(raw: dict) -> tuple[bool, list[str], Optional[InvoiceData]]:
    """
    Run all three validators in order. Stop at first failure group
    to avoid cascading errors from an invalid Pydantic model.

    Returns:
        (all_passed, all_errors, invoice_or_None)
    """
    # Stage 1: schema
    v1 = validate_schema(raw)
    if not v1.passed:
        return False, v1.errors, None

    # Stage 2: types
    v2 = validate_types(v1.invoice)
    if not v2.passed:
        return False, v2.errors, None

    # Stage 3: business rules
    v3 = validate_business_rules(v1.invoice)
    if not v3.passed:
        return False, v3.errors, v1.invoice  # return invoice even if biz rules fail

    return True, [], v1.invoice


print("Validators defined: schema, type, business rules")

## Step 5 — The Correction Loop

This is the core of the pattern. The loop:

1. Attempts extraction (up to `MAX_ATTEMPTS` times).
2. After each attempt, runs all validators.
3. If validation passes, returns the result immediately.
4. If validation fails, feeds the **specific error messages** back to GPT-4o along with the previous JSON. This avoids asking the model to start from scratch — it already has most fields right.
5. After all attempts are exhausted, checks the `confidence` score. If it is below `CONFIDENCE_THRESHOLD`, the result is flagged for human review rather than silently written to the database.

The `@dataclass` result carries the full audit trail: every attempt's raw JSON, the errors found at each step, and whether human review is needed.

In [ ]:
@dataclass
class ExtractionAttempt:
    attempt_num: int
    raw_json: dict
    errors: list[str]
    passed: bool
    elapsed: float


@dataclass
class ExtractionResult:
    invoice: Optional[InvoiceData]
    attempts: list[ExtractionAttempt]
    needs_human_review: bool
    review_reason: str
    total_elapsed: float


async def extract_with_validation_loop(
    image_b64: str,
    page_id: str = "page_1",
    verbose: bool = True,
) -> ExtractionResult:
    """
    Run extraction + validation loop for a single invoice page.

    Args:
        image_b64: Base64-encoded PNG of the invoice page.
        page_id:   Label used for logging (e.g. "page_1", "page_3").
        verbose:   Print progress to stdout.

    Returns:
        ExtractionResult with the best available InvoiceData (may still
        have minor business-rule failures if max attempts exhausted).
    """
    pipeline_start = time.perf_counter()
    attempts: list[ExtractionAttempt] = []
    previous_json_str: Optional[str] = None
    current_errors: Optional[list[str]] = None
    best_invoice: Optional[InvoiceData] = None

    def log(msg: str) -> None:
        if verbose:
            print(f"[{page_id}] {msg}")

    for attempt_num in range(1, MAX_ATTEMPTS + 1):
        log(f"Attempt {attempt_num}/{MAX_ATTEMPTS} — calling {EXTRACTION_MODEL}...")

        raw, elapsed = await extract_invoice(
            image_b64,
            previous_json=previous_json_str,
            errors=current_errors,
        )

        passed, errors, invoice = run_all_validators(raw)

        attempt = ExtractionAttempt(
            attempt_num=attempt_num,
            raw_json=raw,
            errors=errors,
            passed=passed,
            elapsed=elapsed,
        )
        attempts.append(attempt)

        if invoice:
            best_invoice = invoice  # keep latest valid invoice even if biz rules fail

        if passed:
            log(f"Validation PASSED on attempt {attempt_num} ({elapsed:.2f}s)")
            low_confidence = invoice is not None and invoice.confidence < CONFIDENCE_THRESHOLD
            if low_confidence:
                reason = f"confidence={invoice.confidence} < threshold={CONFIDENCE_THRESHOLD}"
                log(f"Flagging for HUMAN REVIEW: {reason}")
            return ExtractionResult(
                invoice=invoice,
                attempts=attempts,
                needs_human_review=low_confidence,
                review_reason=reason if low_confidence else "",
                total_elapsed=time.perf_counter() - pipeline_start,
            )
        else:
            log(f"Validation FAILED (attempt {attempt_num}, {elapsed:.2f}s):")
            for err in errors:
                log(f"  {err}")
            if attempt_num < MAX_ATTEMPTS:
                log("Feeding errors back to model for correction...\n")
                current_errors = errors
                previous_json_str = json.dumps(raw, indent=2)

    # All attempts exhausted — assess confidence
    log(f"All {MAX_ATTEMPTS} attempts exhausted.")

    confidence = best_invoice.confidence if best_invoice else 0
    final_passed = attempts[-1].passed
    needs_review = (not final_passed) or (confidence < CONFIDENCE_THRESHOLD)

    if needs_review:
        reasons = []
        if not final_passed:
            reasons.append(
                f"validation failed after {MAX_ATTEMPTS} attempts"
                f" (errors: {chr(59).join(attempts[-1].errors)})"
            )
        if confidence < CONFIDENCE_THRESHOLD:
            reasons.append(
                f"confidence={confidence} < threshold={CONFIDENCE_THRESHOLD}"
            )
        reason = "; ".join(reasons)
        log(f"Flagging for HUMAN REVIEW: {reason}")

    return ExtractionResult(
        invoice=best_invoice,
        attempts=attempts,
        needs_human_review=needs_review,
        review_reason=reason if needs_review else "",
        total_elapsed=time.perf_counter() - pipeline_start,
    )


print("Extraction loop defined. Max attempts:", MAX_ATTEMPTS)

## Step 6 — Run the Pipeline on the Synthetic Invoice

In [ ]:
result = await extract_with_validation_loop(invoice_b64, page_id="invoice-demo", verbose=True)

In [ ]:
# ── Summary report ─────────────────────────────────────────────────────────
print("=" * 60)
print("EXTRACTION RESULT SUMMARY")
print("=" * 60)
print(f"Total attempts      : {len(result.attempts)}")
print(f"Total elapsed       : {result.total_elapsed:.2f}s")
print(f"Needs human review  : {result.needs_human_review}")
if result.review_reason:
    print(f"Review reason       : {result.review_reason}")
print()

for att in result.attempts:
    status = "PASS" if att.passed else f"FAIL ({len(att.errors)} errors)"
    print(f"  Attempt {att.attempt_num}: {status} — {att.elapsed:.2f}s")
    for err in att.errors:
        print(f"    → {err}")

print()
if result.invoice:
    inv = result.invoice
    print(f"Extracted invoice:")
    print(f"  Vendor          : {inv.vendor}")
    print(f"  Invoice #       : {inv.invoice_number}")
    print(f"  Date            : {inv.date}")
    print(f"  Due Date        : {inv.due_date}")
    print(f"  Currency        : {inv.currency}")
    print(f"  Subtotal        : {inv.subtotal}")
    print(f"  Tax             : {inv.tax}")
    print(f"  Total Amount    : {inv.total_amount}")
    print(f"  Confidence      : {inv.confidence}/5")
    print(f"  Line Items      : {len(inv.line_items)}")
    for i, item in enumerate(inv.line_items):
        print(f"    [{i+1}] {item.description}: {item.quantity} × ${item.unit_price:.2f} = ${item.line_total:.2f}")
    if inv.extraction_notes:
        print(f"  Notes           : {inv.extraction_notes}")
else:
    print("No valid invoice extracted.")

## Step 7 — Verify Against Ground Truth

Compare the extracted values to the known values embedded in the synthetic invoice. This is the end-to-end correctness check.

In [ ]:
def compare_to_ground_truth(inv: InvoiceData) -> None:
    gt = INVOICE_GROUND_TRUTH
    checks = [
        ("vendor",          gt["vendor"],          inv.vendor),
        ("invoice_number",  gt["invoice_number"],  inv.invoice_number),
        ("date",            gt["date"],             inv.date),
        ("due_date",        gt["due_date"],         inv.due_date),
        ("subtotal",        gt["subtotal"],         inv.subtotal),
        ("tax",             gt["tax"],              inv.tax),
        ("total_amount",    gt["total"],            inv.total_amount),
        ("line_item_count", len(gt["line_items"]),  len(inv.line_items)),
    ]

    print(f"{'Field':<20} {'Expected':<20} {'Extracted':<20} {'Match'}")
    print("-" * 75)
    all_pass = True
    for field_name, expected, extracted in checks:
        if isinstance(expected, float):
            match = abs(float(extracted or 0) - expected) < 0.02
        else:
            match = str(extracted) == str(expected)
        mark = "PASS" if match else "FAIL"
        if not match:
            all_pass = False
        print(f"{field_name:<20} {str(expected):<20} {str(extracted):<20} {mark}")

    print("-" * 75)
    print(f"Overall: {'ALL FIELDS MATCH' if all_pass else 'SOME FIELDS DIFFER'}")


if result.invoice:
    compare_to_ground_truth(result.invoice)
else:
    print("Cannot compare — no invoice extracted.")

## Step 8 — Parallel Multi-Page Processing

Real invoices can be multiple pages: a cover page, itemised line items on page 2, and terms on page 3. Processing pages sequentially wastes time. We use `asyncio.gather` to run all pages in parallel — the wall-clock time is determined by the slowest single page, not the sum.

For this demo we simulate two pages by using the same invoice image with a slight label difference. In production, you would convert each PDF page to a PNG and pass each as a separate base64 string.

In [ ]:
async def process_document_pages(
    page_images: list[str],
    document_id: str = "doc_001",
) -> list[ExtractionResult]:
    """
    Process multiple PDF pages in parallel.

    Args:
        page_images: List of base64-encoded PNG strings, one per page.
        document_id: Used as a label prefix for logging.

    Returns:
        List of ExtractionResult objects, one per page.
    """
    tasks = [
        extract_with_validation_loop(
            img,
            page_id=f"{document_id}_p{i+1}",
            verbose=True,
        )
        for i, img in enumerate(page_images)
    ]
    return await asyncio.gather(*tasks)


# Simulate a 2-page document
page_images = [invoice_b64, invoice_b64]  # same image for demo; in production: unique pages

print(f"Processing {len(page_images)} pages in parallel...\n")
multi_page_start = time.perf_counter()
page_results = await process_document_pages(page_images, document_id="invoice-2024-0087")
multi_page_elapsed = time.perf_counter() - multi_page_start

print(f"\nAll {len(page_results)} pages complete in {multi_page_elapsed:.2f}s (parallel)")

for i, pr in enumerate(page_results):
    status = "NEEDS REVIEW" if pr.needs_human_review else "OK"
    vendor = pr.invoice.vendor if pr.invoice else "[no invoice]"
    conf = pr.invoice.confidence if pr.invoice else "?"
    print(f"  Page {i+1}: {status} | vendor={vendor!r} | confidence={conf}/5 | {pr.total_elapsed:.2f}s")

## Step 9 — GPT-4o-mini as a Second-Opinion Validator

For high-value invoices, it is worth running a fast second-opinion check using `gpt-4o-mini`. Rather than re-extracting from the image (which GPT-4o already did well), we ask GPT-4o-mini to audit the extracted JSON for logical consistency. This is purely a text task — no vision tokens — so it is cheap and fast.

The prompt is deliberately adversarial: "Find any inconsistencies or suspicious values." A plain "does this look correct?" produces sycophantic agreement.

In [ ]:
AUDIT_SYSTEM = dedent("""
    You are a financial auditor reviewing an extracted invoice JSON.
    Your ONLY job is to find logical inconsistencies, suspicious values,
    or missing fields that indicate extraction errors.

    Check:
    1. Do line item totals (qty × unit_price) match line_total?
    2. Does subtotal + tax = total_amount?
    3. Are dates logically ordered (invoice date ≤ due date)?
    4. Is the currency code plausible?
    5. Are any amounts implausibly large or small?

    Return JSON: {"issues": ["...", ...], "audit_confidence": 1-5}
    If no issues, return {"issues": [], "audit_confidence": 5}.
    Return ONLY valid JSON. No commentary.
""").strip()


async def audit_with_mini(invoice: InvoiceData) -> dict:
    """Run a fast GPT-4o-mini audit on the extracted JSON."""
    invoice_json = invoice.model_dump_json(indent=2)

    t0 = time.perf_counter()
    response = await client.chat.completions.create(
        model=VALIDATION_MODEL,
        messages=[
            {"role": "system", "content": AUDIT_SYSTEM},
            {"role": "user",   "content": f"Audit this extracted invoice:\n{invoice_json}"},
        ],
        temperature=0.0,
        max_tokens=512,
        response_format={"type": "json_object"},
    )
    elapsed = time.perf_counter() - t0

    audit_result = json.loads(response.choices[0].message.content)
    audit_result["elapsed"] = round(elapsed, 2)
    return audit_result


if result.invoice:
    print(f"Running GPT-4o-mini audit on extracted invoice...")
    audit = await audit_with_mini(result.invoice)
    print(f"Audit complete ({audit['elapsed']}s):")
    print(f"  Issues found      : {len(audit.get('issues', []))}")
    print(f"  Audit confidence  : {audit.get('audit_confidence')}/5")
    for issue in audit.get("issues", []):
        print(f"  → {issue}")
    if not audit.get("issues"):
        print("  No issues found — extraction looks clean.")
else:
    print("Skipping audit — no invoice to audit.")

## Step 10 — Production Considerations

This section does not make API calls. It demonstrates how to extend the pipeline for production use.

In [ ]:
# ── Human review queue (stub) ───────────────────────────────────────────────
# In production this would write to a database table or post to a Slack channel.

from dataclasses import asdict


class HumanReviewQueue:
    """Stub for a human review queue. Replace with your persistence layer."""

    def __init__(self) -> None:
        self._items: list[dict] = []

    def enqueue(self, document_id: str, page: int, result: ExtractionResult) -> None:
        self._items.append({
            "document_id": document_id,
            "page": page,
            "reason": result.review_reason,
            "confidence": result.invoice.confidence if result.invoice else None,
            "attempts": len(result.attempts),
            "errors": result.attempts[-1].errors if result.attempts else [],
        })
        print(f"  [QUEUE] {document_id} page {page} enqueued for human review.")
        print(f"          Reason: {result.review_reason[:80]}...")

    def summary(self) -> None:
        print(f"\nHuman review queue: {len(self._items)} item(s)")
        for item in self._items:
            print(f"  doc={item['document_id']} page={item['page']} "
                  f"confidence={item['confidence']} attempts={item['attempts']}")


queue = HumanReviewQueue()

# Route all page results through the queue check
print("Routing multi-page results...")
for i, pr in enumerate(page_results):
    if pr.needs_human_review:
        queue.enqueue("invoice-2024-0087", i + 1, pr)
    else:
        conf = pr.invoice.confidence if pr.invoice else "?"
        print(f"  Page {i+1}: accepted (confidence={conf}/5, {len(pr.attempts)} attempt(s))")

queue.summary()

In [ ]:
# ── Retry budget analysis ──────────────────────────────────────────────────
# Show how many tokens and seconds the retry loop cost vs single-pass.

def print_attempt_breakdown(results: list[ExtractionResult]) -> None:
    print(f"{'Page':<8} {'Attempts':<10} {'Elapsed':<12} {'Confidence':<12} {'Status'}")
    print("-" * 60)
    for i, r in enumerate(results):
        status = "HUMAN REVIEW" if r.needs_human_review else "ACCEPTED"
        conf = r.invoice.confidence if r.invoice else "-"
        print(
            f"{i+1:<8} "
            f"{len(r.attempts):<10} "
            f"{r.total_elapsed:<12.2f} "
            f"{conf!s:<12} "
            f"{status}"
        )
    total_time = sum(r.total_elapsed for r in results)
    total_attempts = sum(len(r.attempts) for r in results)
    print("-" * 60)
    print(f"Total: {total_attempts} attempts across {len(results)} pages")
    print(f"Wall-clock time (parallel): {multi_page_elapsed:.2f}s")
    print(f"Sum of per-page times     : {total_time:.2f}s")
    print(f"Parallelism speedup       : {total_time / multi_page_elapsed:.1f}x")


print_attempt_breakdown(page_results)

## Summary

This notebook demonstrates a production-grade document extraction pipeline with three layers of defense against silent extraction failures:

### Architecture recap

| Layer | Tool | What it catches |
|---|---|---|
| Schema validation | Pydantic v2 | Missing fields, wrong types, invalid date formats |
| Type/sanity checks | Custom validators | Negative amounts, invalid currency codes |
| Business rules | Arithmetic checks | Line items that don't sum to totals |
| Correction loop | GPT-4o + error feedback | Re-extracts only the broken fields |
| Confidence gating | Model self-report | Routes low-confidence results to human review |
| Second opinion | GPT-4o-mini audit | Cheap adversarial check on extracted JSON |
| Parallel execution | `asyncio.gather` | Multi-page processing at full throughput |

### Key implementation patterns

- **Feed specific errors, not the full schema**: The correction prompt includes the exact error message from validation, not a restatement of the schema. This directs the model's attention precisely.
- **`response_format={"type": "json_object"}`**: Forces well-formed JSON on every call, eliminating markdown fence stripping and partial-JSON errors.
- **Temperature 0**: Deterministic extraction. Variability between retries comes from seeing the error feedback, not from sampling.
- **Keep the best invoice across attempts**: If attempt 2 passes schema but fails a business rule, we keep it rather than discarding it. It is better data than nothing.
- **Confidence as a routing signal**: The model's self-reported confidence is not used to skip validation — validation runs regardless. It is used only to decide whether a result that exhausted its retry budget should go to humans or be accepted with caveats.

### Extending this pattern

1. **PDF-to-image conversion**: Use `pdf2image` (wraps `poppler`) or `pymupdf` to convert PDF pages to PNGs before passing to this pipeline.
2. **Multi-document batching**: Wrap `process_document_pages` in a semaphore (`asyncio.Semaphore`) to rate-limit concurrent API calls against your OpenAI tier.
3. **Schema versioning**: Store `InvoiceData.model_json_schema()` alongside extracted records so you can replay extractions when the schema changes.
4. **Fine-tuned validation**: For domain-specific documents (medical claims, legal contracts), replace the business-rule validator with domain-specific checks — CPT code ranges, contract clause presence checks, etc.
5. **Structured outputs (beta)**: OpenAI's `response_format={"type": "json_schema", "json_schema": {...}}` endpoint can enforce the Pydantic schema at the API level, reducing schema validation failures to near zero.